# NeuralFoil vs XFLR5 on the same Re/alpha grid

This notebook is standalone and does:
1. Build Baseline/Optimised airfoils from CST parameters
2. Read Reynolds/alpha grid from XFLR5 comparison CSV
3. Run NeuralFoil for both airfoils on the same grid
4. Save NeuralFoil results to CSV
5. Provide an interactive Reynolds selector to compare CL, CD, CL/CD
   (same color per airfoil, different line style per simulation source).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import yaml

from glider_optimization.utils.cu_kulfan_airfoil import get_aero_from_kulfan_parameters_cuda

ROOT = Path('/Users/gherardi/Documents/GitHub/glider_optimization')
XFLR5_GRID_CSV = ROOT / 'artifacts' / 'xfoil' / 'Results' / 'xflr5_baseline_vs_optimised_grid.csv'
NF_OUT_CSV = ROOT / 'artifacts' / 'xfoil' / 'Results' / 'neuralfoil_baseline_optimised_grid.csv'
CFG_PATH = ROOT / 'conf' / 'test.yaml'

if not XFLR5_GRID_CSV.exists():
    raise FileNotFoundError(f'Missing input CSV: {XFLR5_GRID_CSV}')

cfg = yaml.safe_load(CFG_PATH.read_text()) if CFG_PATH.exists() else {}
nf_cfg = (cfg or {}).get('neuralFoilSampling', {}) or {}
MODEL_SIZE = str(nf_cfg.get('neuralFoil_size', 'xxxlarge'))
DEVICE = 'cpu'

print('Input grid CSV :', XFLR5_GRID_CSV)
print('Output NF CSV  :', NF_OUT_CSV)
print('NeuralFoil model:', MODEL_SIZE)

Input grid CSV : /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/xflr5_baseline_vs_optimised_grid.csv
Output NF CSV  : /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/neuralfoil_baseline_optimised_grid.csv
NeuralFoil model: xxxlarge


In [2]:
# 1) Baseline / Optimised airfoils from CST parameters (copied from dat_from_cst.ipynb)
baseline_params = {
    'name': 'Baseline',
    'lower_weights': [-0.16965146, -0.09364138, -0.06345896, -0.0067966, -0.0902447, 0.02081845, -0.03575216, -0.00223623],
    'upper_weights': [0.18109497, 0.21268419, 0.28098503, 0.24864887, 0.2402814, 0.27262843, 0.25776474, 0.27817638],
    'leading_edge_weight': 0.10647,
    'TE_thickness': 0.00257,
}

optimised_params = {
    'name': 'Optimised',
    'lower_weights': [-0.10264578461647034, -0.002956927753984928, 0.04788283631205559, 0.14223960041999817, 0.15873770415782928, 0.28605443239212036, 0.23426659405231476, 0.31003499031066895],
    'upper_weights': [0.23543250560760495, 0.16666221618652344, 0.11111072450876236, 0.19317159056663513, 0.20873770117759705, 0.3360544443130493, 0.2842665910720825, 0.3600350022315979],
    'leading_edge_weight': 0.10302258282899857,
    'TE_thickness': 0.009999999776482582,
}

airfoils = [baseline_params, optimised_params]
print('Airfoils loaded:', [a['name'] for a in airfoils])

Airfoils loaded: ['Baseline', 'Optimised']


In [3]:
# 2) Extract Reynolds + alpha points from XFLR5 comparison CSV
df_x = pd.read_csv(XFLR5_GRID_CSV)
grid = (
    df_x[['Re', 'alpha']]
    .dropna()
    .drop_duplicates()
    .astype({'Re': int})
    .sort_values(['Re', 'alpha'])
    .reset_index(drop=True)
)

re_values = sorted(grid['Re'].unique().tolist())
alpha_values = sorted(grid['alpha'].unique().tolist())

print('Grid points:', len(grid))
print('Unique Re count:', len(re_values))
print('Unique alpha count:', len(alpha_values))
print('Re range:', re_values[0], '->', re_values[-1])
print('Alpha range:', alpha_values[0], '->', alpha_values[-1])

Grid points: 3872
Unique Re count: 32
Unique alpha count: 121
Re range: 160 -> 99940
Alpha range: -30.0 -> 30.0


In [4]:
# 3) Run NeuralFoil on the same grid for both airfoils
def eval_airfoil_on_grid(params: dict, grid_df: pd.DataFrame, model_size: str = 'xxxlarge', device: str = 'cpu') -> pd.DataFrame:
    B = len(grid_df)
    alpha_t = torch.as_tensor(grid_df['alpha'].to_numpy(), dtype=torch.float32)
    re_t = torch.as_tensor(grid_df['Re'].to_numpy(), dtype=torch.float32)

    upper = torch.as_tensor(params['upper_weights'], dtype=torch.float32).unsqueeze(0).repeat(B, 1)
    lower = torch.as_tensor(params['lower_weights'], dtype=torch.float32).unsqueeze(0).repeat(B, 1)
    le = torch.full((B,), float(params['leading_edge_weight']), dtype=torch.float32)
    te = torch.full((B,), float(params['TE_thickness']), dtype=torch.float32)

    kulfan = {
        'upper_weights_cuda': upper,
        'lower_weights_cuda': lower,
        'leading_edge_weight_cuda': le,
        'TE_thickness_cuda': te,
    }

    aero = get_aero_from_kulfan_parameters_cuda(
        kulfan_parameters_cuda=kulfan,
        alpha=alpha_t,
        Re=re_t,
        model_size=model_size,
        device=device,
    )

    out = grid_df.copy()
    out['airfoil'] = params['name']
    out['nf_CL'] = aero['CL'].detach().cpu().numpy()
    out['nf_CD'] = aero['CD'].detach().cpu().numpy()
    out['nf_CM'] = aero['CM'].detach().cpu().numpy()
    conf = aero.get('analysis_confidence', None)
    out['nf_confidence'] = conf.detach().cpu().numpy() if conf is not None else np.nan
    out['nf_CLCD'] = out['nf_CL'] / out['nf_CD'].replace(0.0, np.nan)
    return out

nf_parts = [eval_airfoil_on_grid(a, grid, model_size=MODEL_SIZE, device=DEVICE) for a in airfoils]
df_nf = pd.concat(nf_parts, ignore_index=True)

print('NeuralFoil rows:', len(df_nf))
df_nf.head()

NeuralFoil rows: 7744


,Re,alpha,airfoil,nf_CL,nf_CD,nf_CM,nf_confidence,nf_CLCD
0,160,-30.0,Baseline,-0.692612,0.473103,0.049299,1.006204e-13,-1.463978
1,160,-29.5,Baseline,-0.687907,0.469926,0.047325,7.123258e-12,-1.463862
2,160,-29.0,Baseline,-0.683178,0.466715,0.045274,2.703181e-10,-1.463801
3,160,-28.5,Baseline,-0.678425,0.463481,0.043142,5.879884e-09,-1.463761
4,160,-28.0,Baseline,-0.673655,0.460236,0.040927,7.808714e-08,-1.463717


In [5]:
# 4) Save NeuralFoil results as CSV
NF_OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_nf.to_csv(NF_OUT_CSV, index=False)
print('Saved NeuralFoil CSV:', NF_OUT_CSV)

Saved NeuralFoil CSV: /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xfoil/Results/neuralfoil_baseline_optimised_grid.csv


In [ ]:
# 5) Standalone interactive comparison plot (Re dropdown)
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Reload from CSVs so this cell is standalone
df_x = pd.read_csv(XFLR5_GRID_CSV)
df_nf_plot = pd.read_csv(NF_OUT_CSV)

# Build XFLR5 long format (airfoil-specific columns -> tidy table)
x_b = df_x[['Re', 'alpha', 'baseline_CL', 'baseline_CD']].copy()
x_b['airfoil'] = 'Baseline'
x_b = x_b.rename(columns={'baseline_CL': 'CL', 'baseline_CD': 'CD'})

x_o = df_x[['Re', 'alpha', 'optimised_CL', 'optimised_CD']].copy()
x_o['airfoil'] = 'Optimised'
x_o = x_o.rename(columns={'optimised_CL': 'CL', 'optimised_CD': 'CD'})

df_x_long = pd.concat([x_b, x_o], ignore_index=True)
df_x_long['CLCD'] = df_x_long['CL'] / df_x_long['CD'].replace(0.0, np.nan)

# NeuralFoil tidy selection
df_nf_long = df_nf_plot[['Re', 'alpha', 'airfoil', 'nf_CL', 'nf_CD', 'nf_CLCD']].copy()
df_nf_long = df_nf_long.rename(columns={'nf_CL': 'CL', 'nf_CD': 'CD', 'nf_CLCD': 'CLCD'})

re_values = sorted(df_x_long['Re'].dropna().astype(int).unique().tolist())
if not re_values:
    raise ValueError('No Reynolds numbers found in XFLR5 CSV.')

AIRFOIL_COLORS = {'Baseline': 'tab:blue', 'Optimised': 'tab:orange'}
SIM_STYLES = {'XFLR5': '-', 'NeuralFoil': '--'}

def add_missing_markers(ax, alpha, y, color, label):
    miss = alpha[np.isnan(y)]
    if len(miss) == 0:
        return
    y0, y1 = ax.get_ylim()
    y_mark = y0 + 0.05 * (y1 - y0)
    ax.scatter(miss, np.full_like(miss, y_mark, dtype=float), marker='x', s=28, color=color, alpha=0.9, label=label, zorder=4)

def plot_metric(ax, re_sel, metric):
    for airfoil in ['Baseline', 'Optimised']:
        color = AIRFOIL_COLORS[airfoil]

        sx = df_x_long[(df_x_long['Re'] == re_sel) & (df_x_long['airfoil'] == airfoil)].sort_values('alpha')
        sn = df_nf_long[(df_nf_long['Re'] == re_sel) & (df_nf_long['airfoil'] == airfoil)].sort_values('alpha')

        ax.plot(sx['alpha'].to_numpy(), sx[metric].to_numpy(), SIM_STYLES['XFLR5'], color=color, linewidth=2, label=f'{airfoil} - XFLR5')
        ax.plot(sn['alpha'].to_numpy(), sn[metric].to_numpy(), SIM_STYLES['NeuralFoil'], color=color, linewidth=2, label=f'{airfoil} - NeuralFoil')

        add_missing_markers(ax, sx['alpha'].to_numpy(dtype=float), sx[metric].to_numpy(dtype=float), color, f'{airfoil} XFLR5 missing')
        add_missing_markers(ax, sn['alpha'].to_numpy(dtype=float), sn[metric].to_numpy(dtype=float), color, f'{airfoil} NeuralFoil missing')

    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')

def plot_for_re(selected_re):
    re_sel = int(selected_re)

    fig, axes = plt.subplots(3, 1, figsize=(11, 12), sharex=True)
    plot_metric(axes[0], re_sel, 'CL')
    plot_metric(axes[1], re_sel, 'CD')
    plot_metric(axes[2], re_sel, 'CLCD')

    axes[0].set_title(f'XFLR5 vs NeuralFoil at Re={re_sel}')
    axes[2].set_xlabel('alpha (deg)')
    plt.tight_layout()
    plt.show()

re_dropdown = widgets.Dropdown(
    options=re_values,
    value=re_values[0],
    description='Re:',
    layout=widgets.Layout(width='300px')
)

ui = widgets.interactive_output(plot_for_re, {'selected_re': re_dropdown})
display(re_dropdown, ui)

Dropdown(description='Re:', layout=Layout(width='300px'), options=(160, 641, 1597, 3020, 4896, 7206, 9930, 130…

Output()